### FastAPI Real-World Usage
Pydantic-Aware Descriptor for Shared Validation Logic

In [ ]:
# schemas/descriptors.py
import re
from pydantic import field_validator
from functools import wraps


def slug_validator(field_name: str):
    """
    Returns a reusable Pydantic field_validator for slug fields.
    Not a descriptor, but same DRY principle — factory pattern.
    """
    pattern = re.compile(r'^[a-z0-9]+(?:-[a-z0-9]+)*$')

    @field_validator(field_name)
    @classmethod
    def validate_slug(cls, v: str) -> str:
        v = v.strip().lower()
        if not pattern.match(v):
            raise ValueError(
                f"{field_name!r} must be a valid slug (lowercase, hyphens only)"
            )
        return v
    return validate_slug


# schemas/post.py
from pydantic import BaseModel
from .descriptors import slug_validator

class PostCreate(BaseModel):
    title: str
    slug:  str

    _validate_slug = slug_validator("slug")


class CategoryCreate(BaseModel):
    name: str
    slug: str

    _validate_slug = slug_validator("slug")  # ← reused, not rewritten

Descriptor for Request-Scoped Caching (Service Layer)

In [ ]:
# services/base.py
class ServiceCache:
    """
    Non-data descriptor for lazy-loading and caching expensive
    service-layer dependencies within a request scope.
    """
    def __init__(self, factory):
        self.factory = factory
        self.name    = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.name not in obj.__dict__:
            obj.__dict__[self.name] = self.factory(obj)
        return obj.__dict__[self.name]


# services/order_service.py
class OrderService:
    def __init__(self, db_session, user_id: int):
        self.db_session = db_session
        self.user_id    = user_id

    @ServiceCache
    def user(self):
        """Fetched once per service instance — cached on first access."""
        return self.db_session.query(User).filter_by(id=self.user_id).first()

    @ServiceCache
    def cart(self):
        """Fetched once — subsequent accesses return cached instance."""
        return self.db_session.query(Cart).filter_by(user_id=self.user_id).first()

    def checkout(self) -> dict:
        user = self.user   # DB hit → cached
        cart = self.cart   # DB hit → cached
        _    = self.user   # no DB hit — from __dict__
        _    = self.cart   # no DB hit — from __dict__
        return {"user": user.email, "total": cart.total}